# Análisis de Calidad de Datos
Diagnóstico rápido del estado del dataset:
1. Ficheros XLSX incompletos (< 2880 filas)
2. NaN por variable en el CSV final
3. Mapa de huecos por mes y variable

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATASET_DIR = Path('Dataset')
CSV_PATH    = Path('data/2024_03_06-2025_03_05_1min.csv')
FILAS_DIA   = 2880  # 30s × 2 × 60min × 24h

## 1. Estado de los ficheros XLSX

In [ ]:
ficheros = sorted(DATASET_DIR.glob('AGROCONNECT_*.xlsx'))
print(f'Total ficheros: {len(ficheros)}')

registros = []
for f in ficheros:
    try:
        df = pd.read_excel(f, usecols=['FECHA'])
        n = len(df)
        fecha_str = f.stem.split('_')[1][:8]  # YYYYMMDD
        fecha = pd.to_datetime(fecha_str, format='%Y%m%d')
        registros.append({'fichero': f.name, 'fecha': fecha, 'filas': n, 'pct': round(n / FILAS_DIA * 100, 1)})
    except Exception as e:
        registros.append({'fichero': f.name, 'fecha': pd.NaT, 'filas': -1, 'pct': 0})

df_files = pd.DataFrame(registros).sort_values('fecha').reset_index(drop=True)

incompletos = df_files[df_files['filas'] < FILAS_DIA * 0.9]  # menos del 90%
print(f'\nFicheros con < 90% de datos ({int(FILAS_DIA*0.9)} filas): {len(incompletos)}')
print(incompletos[['fichero', 'fecha', 'filas', 'pct']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
colores = ['#d32f2f' if p < 50 else '#ff9800' if p < 90 else '#388e3c' for p in df_files['pct']]
ax.bar(df_files['fecha'], df_files['pct'], color=colores, width=0.8)
ax.axhline(90, color='orange', linestyle='--', linewidth=1, label='90%')
ax.axhline(50, color='red',    linestyle='--', linewidth=1, label='50%')
ax.set_ylabel('% datos respecto a día completo')
ax.set_title('Completitud de cada fichero XLSX')
ax.legend()
plt.tight_layout()
plt.show()

## 2. NaN por variable en el CSV final

In [ ]:
df = pd.read_csv(CSV_PATH)
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
print(f'CSV cargado: {len(df):,} filas  |  {df["Fecha"].iloc[0]} → {df["Fecha"].iloc[-1]}')

variables = [c for c in df.columns if c != 'Fecha']
resumen = pd.DataFrame({
    'nulos':  df[variables].isnull().sum(),
    'pct':    (df[variables].isnull().mean() * 100).round(2)
}).sort_values('pct', ascending=False)

print('\nNaN por variable:')
print(resumen.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colores = ['#d32f2f' if p > 20 else '#ff9800' if p > 5 else '#388e3c' for p in resumen['pct']]
ax.barh(resumen.index, resumen['pct'], color=colores)
ax.axvline(5,  color='orange', linestyle='--', linewidth=1, label='5%')
ax.axvline(20, color='red',    linestyle='--', linewidth=1, label='20%')
ax.set_xlabel('% NaN')
ax.set_title('Porcentaje de NaN por variable')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Mapa de huecos: NaN por mes y variable

In [ ]:
df['mes'] = df['Fecha'].dt.to_period('M').astype(str)

heatmap_data = df.groupby('mes')[variables].apply(lambda g: g.isnull().mean() * 100)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    heatmap_data.T,
    ax=ax,
    cmap='RdYlGn_r',
    vmin=0, vmax=100,
    annot=True, fmt='.0f',
    linewidths=0.5,
    cbar_kws={'label': '% NaN'}
)
ax.set_title('% NaN por variable y mes')
ax.set_xlabel('Mes')
ax.set_ylabel('Variable')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Días con más NaN en el CSV

In [ ]:
df['dia'] = df['Fecha'].dt.date
nan_diario = df.groupby('dia')[variables].apply(lambda g: g.isnull().mean() * 100)
nan_diario['media'] = nan_diario.mean(axis=1)

peores = nan_diario[nan_diario['media'] > 50].sort_values('media', ascending=False)
print(f'Días con >50% NaN de media: {len(peores)}')
if not peores.empty:
    print(peores[['media'] + variables].round(1).to_string())